# Wind Data Exploration

Explore raw and processed NetCDF data for a case study before training.

**Usage:** Set `CASE_STUDY` below to point at your case study directory.

In [ ]:
import sys
from pathlib import Path

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import yaml

%matplotlib inline

# --- Configuration ---
CASE_STUDY = '../case_studies/sf_bay_conus404'

case_dir = Path(CASE_STUDY)
raw_dir = case_dir / 'data' / 'raw'
processed_dir = case_dir / 'data' / 'processed'

# Load preprocessing config to get file mappings
with open(case_dir / 'configs' / 'preprocessing.yaml') as f:
    preproc_config = yaml.safe_load(f)

with open(case_dir / 'configs' / 'training.yaml') as f:
    train_config = yaml.safe_load(f)

print(f'Case study: {case_dir.name}')
print(f'Raw data: {raw_dir}')
print(f'Processed data: {processed_dir}')

## 1. List raw data files

In [ ]:
file_dict = preproc_config['file_dict']

print('Variable -> File')
print('-' * 70)
for var_key, filename in file_dict.items():
    filepath = raw_dir / filename
    status = 'OK' if filepath.exists() else 'MISSING'
    print(f'  [{status}] {var_key}: {filename}')

## 2. Inspect a raw file

Open one of the raw NetCDF files and examine its structure.

In [ ]:
# Pick the first available file
sample_var, sample_file = next(iter(file_dict.items()))
sample_path = raw_dir / sample_file

if sample_path.exists():
    ds = xr.open_dataset(sample_path)
    print(f'File: {sample_file}')
    print(f'Variables: {list(ds.data_vars)}')
    print(f'Coordinates: {list(ds.coords)}')
    print(f'Dimensions: {dict(ds.dims)}')
    print()
    print(ds)
else:
    print(f'File not found: {sample_path}')
    print('Place your raw NetCDF data in the raw data directory and re-run.')
    ds = None

## 3. Spatial overview

Plot the first timestep of the first variable to see the spatial extent.

In [ ]:
if ds is not None:
    var_name = list(ds.data_vars)[0]
    data = ds[var_name].isel(time=0)

    fig, ax = plt.subplots(figsize=(10, 8))
    data.plot(ax=ax, cmap='RdBu_r')
    ax.set_title(f'{sample_var} ({var_name}) - First Timestep')
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()
else:
    print('No data loaded.')

## 4. Statistics across all raw variables

In [ ]:
print(f'{"Variable":<20} {"Shape":<30} {"Mean":>10} {"Std":>10} {"Min":>10} {"Max":>10}')
print('-' * 90)

for var_key, filename in file_dict.items():
    fpath = raw_dir / filename
    if not fpath.exists():
        print(f'{var_key:<20} FILE MISSING')
        continue
    tmp = xr.open_dataset(fpath)
    vname = list(tmp.data_vars)[0]
    arr = tmp[vname]
    print(f'{var_key:<20} {str(arr.shape):<30} '
          f'{float(arr.mean()):>10.3f} {float(arr.std()):>10.3f} '
          f'{float(arr.min()):>10.3f} {float(arr.max()):>10.3f}')
    tmp.close()

## 5. Time series at grid center

Plot a time series at the center of the domain for a quick sanity check.

In [ ]:
if ds is not None:
    var_name = list(ds.data_vars)[0]
    arr = ds[var_name]

    # Select center of the spatial domain
    y_dim = [d for d in arr.dims if d != 'time'][0]
    x_dim = [d for d in arr.dims if d != 'time'][1]
    center_y = arr.sizes[y_dim] // 2
    center_x = arr.sizes[x_dim] // 2

    ts = arr.isel({y_dim: center_y, x_dim: center_x})

    fig, ax = plt.subplots(figsize=(14, 4))
    ts.plot(ax=ax)
    ax.set_title(f'{var_name} at grid center ({y_dim}={center_y}, {x_dim}={center_x})')
    ax.set_xlabel('Time')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No data loaded.')

## 6. Compare ERA5 vs CONUS404 for a wind variable

Side-by-side comparison of low-res input and high-res target.

In [ ]:
# Find an ERA5/CONUS404 pair from the config
pairs = train_config.get('variable_pairs', {})
pair_name = next(iter(pairs))  # e.g. 'wind_u'
pair = pairs[pair_name]

low_key = pair['low_res']
high_key = pair['high_res']

low_file = raw_dir / file_dict.get(low_key, '')
high_file = raw_dir / file_dict.get(high_key, '')

if low_file.exists() and high_file.exists():
    ds_low = xr.open_dataset(low_file)
    ds_high = xr.open_dataset(high_file)

    low_var = list(ds_low.data_vars)[0]
    high_var = list(ds_high.data_vars)[0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ds_low[low_var].isel(time=0).plot(ax=axes[0], cmap='RdBu_r')
    axes[0].set_title(f'{low_key} (ERA5)')
    axes[0].set_aspect('equal')

    ds_high[high_var].isel(time=0).plot(ax=axes[1], cmap='RdBu_r')
    axes[1].set_title(f'{high_key} (CONUS404)')
    axes[1].set_aspect('equal')

    fig.suptitle(f'{pair_name} - First Timestep', fontsize=14)
    plt.tight_layout()
    plt.show()

    ds_low.close()
    ds_high.close()
else:
    print(f'Pair files not found for {pair_name}.')
    print(f'  Low-res: {low_file} (exists={low_file.exists()})')
    print(f'  High-res: {high_file} (exists={high_file.exists()})')

## 7. Inspect processed data (after preprocessing)

If you have already run `scripts/preprocess_training.py`, inspect the processed train/val/test splits.

In [ ]:
for split in ['train', 'val', 'test']:
    fpath = processed_dir / f'{split}.nc'
    if fpath.exists():
        tmp = xr.open_dataset(fpath)
        print(f'{split}: {len(tmp.time)} timesteps, vars={list(tmp.data_vars)}')
        tmp.close()
    else:
        print(f'{split}: not yet created (run scripts/preprocess_training.py first)')

stats_path = processed_dir / 'normalization_stats.pkl'
if stats_path.exists():
    import pickle
    with open(stats_path, 'rb') as f:
        stats = pickle.load(f)
    print(f'\nNormalization stats for {len(stats)} variables:')
    for var, s in stats.items():
        print(f'  {var}: mean={s["mean"]:.4f}, std={s["std"]:.4f}')
else:
    print('\nNormalization stats: not yet created')